In [1]:
import pandas as pd
import numpy as np
import duckdb

#### **How We Loaded in the Dataset: *US Airline Flight Routes and Fares 1993-2024.csv***
**Map of data and loading in CSV file:**\
Our dataset is very large, existing with an original 245955 rows and 23 columns. Due to this, our dataset was taking up an enormous amount of space and processing time. In order to reduce this, we essentially created a map of each column's expected datatype, as to prevent pandas needing to infer the data type per column. This saves space while loading in the dataset, and also allows it to load quicker.

In [2]:
dtype_map = {
    "tbl": "object",
    "Year": "int64",
    "quarter": "int64",
    "citymarketid_1": "int64",
    "citymarketid_2": "int64",
    "city1": "object",
    "city2": "object",
    "airportid_1": "int64",
    "airportid_2": "int64",
    "airport_1": "object",
    "airport_2": "object",
    "nsmiles": "int64",
    "passengers": "int64",
    "fare": "float64",
    "carrier_lg": "object",
    "large_ms": "float64",
    "fare_lg": "float64",
    "carrier_low": "object",
    "lf_ms": "float64",
    "fare_low": "float64",
    "Geocoded_City1": "object",
    "Geocoded_City2": "object",
    "tbl1apk": "object"
}

flight_df = pd.read_csv(
    'US Airline Flight Routes and Fares 1993-2024.csv', 
    dtype=dtype_map)

print(flight_df.shape)
flight_df.info()
flight_df.head()

(245955, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245955 entries, 0 to 245954
Data columns (total 23 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   tbl             245955 non-null  object 
 1   Year            245955 non-null  int64  
 2   quarter         245955 non-null  int64  
 3   citymarketid_1  245955 non-null  int64  
 4   citymarketid_2  245955 non-null  int64  
 5   city1           245955 non-null  object 
 6   city2           245955 non-null  object 
 7   airportid_1     245955 non-null  int64  
 8   airportid_2     245955 non-null  int64  
 9   airport_1       245955 non-null  object 
 10  airport_2       245955 non-null  object 
 11  nsmiles         245955 non-null  int64  
 12  passengers      245955 non-null  int64  
 13  fare            245955 non-null  float64
 14  carrier_lg      244415 non-null  object 
 15  large_ms        244415 non-null  float64
 16  fare_lg         244415 non-null  float64
 1

,tbl,Year,quarter,citymarketid_1,citymarketid_2,city1,city2,airportid_1,airportid_2,airport_1,...,fare,carrier_lg,large_ms,fare_lg,carrier_low,lf_ms,fare_low,Geocoded_City1,Geocoded_City2,tbl1apk
0,Table1a,2021,3,30135,33195,"Allentown/Bethlehem/Easton, PA","Tampa, FL (Metropolitan Area)",10135,14112,ABE,...,81.43,G4,1.0000,81.43,G4,1.0000,81.43,NaN,NaN,202131013514112ABEPIE
1,Table1a,2021,3,30135,33195,"Allentown/Bethlehem/Easton, PA","Tampa, FL (Metropolitan Area)",10135,15304,ABE,...,208.93,DL,0.4659,219.98,UA,0.1193,154.11,NaN,NaN,202131013515304ABETPA
2,Table1a,2021,3,30140,30194,"Albuquerque, NM","Dallas/Fort Worth, TX",10140,11259,ABQ,...,184.56,WN,0.9968,184.44,WN,0.9968,184.44,NaN,NaN,202131014011259ABQDAL
3,Table1a,2021,3,30140,30194,"Albuquerque, NM","Dallas/Fort Worth, TX",10140,11298,ABQ,...,182.64,AA,0.9774,183.09,AA,0.9774,183.09,NaN,NaN,202131014011298ABQDFW
4,Table1a,2021,3,30140,30466,"Albuquerque, NM","Phoenix, AZ",10140,14107,ABQ,...,177.11,WN,0.6061,184.49,AA,0.3939,165.77,NaN,NaN,202131014014107ABQPHX


**Finding Missing Values:**\
While we know that missing values can be handled by a NaN value, it is always better to not have them. These missing values can cause errors in our statistical analysis, especially in a machine learning model

In [3]:
missing_values_per_column = flight_df.isnull().sum()
print(missing_values_per_column)

tbl                   0
Year                  0
quarter               0
citymarketid_1        0
citymarketid_2        0
city1                 0
city2                 0
airportid_1           0
airportid_2           0
airport_1             0
airport_2             0
nsmiles               0
passengers            0
fare                  0
carrier_lg         1540
large_ms           1540
fare_lg            1540
carrier_low        1612
lf_ms              1612
fare_low           1612
Geocoded_City1    39206
Geocoded_City2    39206
tbl1apk               0
dtype: int64


**Dropping Unnecessary Columns for the *flight_df*:**\
Within this dataset, several columns contain identifiers or redundant information that are not relevant to our analysis. Since our research focuses on how distance, passenger volume, and carrier type affect airfare, we will remove columns that do not contribute to those variables.

We are dropping:
- tbl: table label not needed for analysis
- citymarketid_1, citymarketid_2: internal market identifiers
- airportid_1, airportid_2: duplicate information since we are keeping airport codes instead
- Geocoded_City1, Geocoded_City2: geocoded fields with redundant data

These columns do not give insight into airfare patterns and will be removed to simplify the dataset.


In [4]:
flight_df = flight_df.drop(
    ['tbl', 'airportid_1',
     'airportid_2',
     'city1',
     'city2',
     'airport_1',
     'airport_2',
     'citymarketid_1',
     'citymarketid_2',
     'Geocoded_City1',
     'Geocoded_City2',
     'tbl1apk'],
    axis=1)
flight_df = flight_df.drop(
    ['carrier_lg',
      'large_ms',
        'fare_lg',
     'lf_ms',
          'carrier_low',
              'fare_low'],
      axis=1)
flight_df.head()

,Year,quarter,nsmiles,passengers,fare
0,2021,3,970,180,81.43
1,2021,3,970,19,208.93
2,2021,3,580,204,184.56
3,2021,3,580,264,182.64
4,2021,3,328,398,177.11


We check again to make sure we have no missing values.

In [5]:
missing_values_per_column = flight_df.isnull().sum()
print(missing_values_per_column)

Year          0
quarter       0
nsmiles       0
passengers    0
fare          0
dtype: int64


**Removing Invalid Values in Columns:**

Some rows could contain impossible or invalid values (such as a fare of 0, negative distance,
or a passenger count of 0). These values are not realistic for actual flight routes and
would distort statistical models. Therefore, we remove rows where any of the following
conditions occur:

- fare ≤ 0  
- nsmiles ≤ 0  
- passengers ≤ 0  

In [6]:
flight_df = flight_df[
    (flight_df["fare"] > 0) &
    (flight_df["nsmiles"] > 0) &
    (flight_df["passengers"] > 0)
]

print("Shape after removing invalid values:", flight_df.shape)


Shape after removing invalid values: (238516, 5)


**Dropping Remaining Missing Values:**

Although we previously inspected missing values, we need to make sure that no NA values remain in numeric variables before analysis.

We drop rows where fare, nsmiles, or passengers contain NA.

In [7]:
flight_df = flight_df.dropna(subset=["fare", "nsmiles", "passengers"])

print("Shape after dropping NA rows:", flight_df.shape)

Shape after dropping NA rows: (238516, 5)


#### **How we loaded in the Dataset: *CPI_data - Sheet1.csv***
This data represents the *Consumer Price Index (CPI)* per year from 1960 to 2024 for most countries; however, we will clean the data to contain only the CPI data from 1993 to 2024 for the United States. Since this data frame is relatively small- in comparison to our *flights_df*-we are able to load it in as a regular data frame using pandas.

**Loading in the CPI csv file:**

In [8]:
cpi_per_year_df = pd.read_csv('CPI_data - Sheet1 .csv')

print('This is the shape of cpi_per_year_df: ' 
      + str(cpi_per_year_df.shape))
cpi_per_year_df.head()

This is the shape of cpi_per_year_df: (266, 69)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Aruba,ABW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,0.474764,-0.931196,-1.028282,3.626041,4.257462,NaN,NaN,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,5.245878,6.596505,6.399343,4.720805,4.644967,5.405162,7.240978,10.773751,7.126975,4.425471
2,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.661709,4.383892,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186
3,Africa Western and Central,AFW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,2.130817,1.487416,1.725486,1.784050,1.983092,2.490378,3.745568,7.930929,5.489286,3.615966
4,Angola,AGO,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,...,9.355972,30.694415,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495


**Selecting United States Data:**\
Here we are selecting only the row with United States data from the *cpi_per_year_df*, and saving it as *cpi_per_year_us*.

In [9]:
# This data is a series now, since it is only one row
cpi_per_year_us = cpi_per_year_df.iloc[251] 

print('This is the shape of cpi_per_year_us: ' +
       str(cpi_per_year_us.shape))
print(cpi_per_year_us)

This is the shape of cpi_per_year_us: (69,)
Country Name                              United States
Country Code                                        USA
Indicator Name    Inflation, consumer prices (annual %)
Indicator Code                           FP.CPI.TOTL.ZG
1960                                           1.457976
                                  ...                  
2020                                           1.233584
2021                                           4.697859
2022                                             8.0028
2023                                           4.116338
2024                                           2.949525
Name: 251, Length: 69, dtype: object


**Finding Missing Values:**\
While we know that missing values can be handled by a NaN value, it is always better to not have them. These missing values can cause errors in our statistical analysis, especially in a machine learning model

*Please note that we only have a series, so we are only checking one row of data*

In [10]:
missing_values = cpi_per_year_us.isnull().sum()
print(missing_values)

0


**Dropping columns:**

 Now we are dropping the columns we do not need from *cpi_per_year_us*:
- *Country Name*: We do not need this as it is assumed our scope is only referring to United States Data
- *Country Code*: We do not need this because our scope is only United States data
- *Indicator Name*: We do not need this because our scope is only United States data
- *Indicator Code*: We do not need this because our scope is only United States data
- *Years from 1960 to 1992 inclusive*: We do not need to analyze data prior to 1993.

In [11]:
years_to_drop = np.arange(1960, 1993).astype(str)
cpi_per_year_us = cpi_per_year_us.drop(years_to_drop)
cpi_per_year_us = cpi_per_year_us.drop(
    ['Country Name',
      'Country Code',
        'Indicator Name',
          'Indicator Code',])

print(
    'This is the shape of cpi_per_year_us after dropping uneeded columns: ' 
    + str(cpi_per_year_us.shape))

This is the shape of cpi_per_year_us after dropping uneeded columns: (32,)


**Editing cpi_per_year_us:**

Here we are editing *cpi_per_year_us* to become a data frame rather than a series. In this process, *cpi_per_year_us* is melted, and so we also add appropriate column titles by resetting the idices, and thus adding the titles we want.

In [12]:
cpi_per_year_us_df = cpi_per_year_us.to_frame()
cpi_per_year_us_df = cpi_per_year_us_df.reset_index()
cpi_per_year_us_df.columns = ['Year', 'CPI_per_year']
cpi_per_year_us_df['CPI_per_year'] = cpi_per_year_us_df['CPI_per_year']

print(
    'This is the shape of cpi_per_year_us after dropping uneeded columns: '
      + str(cpi_per_year_us.shape))
cpi_per_year_us_df.head()

#print(cpi_per_year_us_df)

This is the shape of cpi_per_year_us after dropping uneeded columns: (32,)


,Year,CPI_per_year
0,1993,2.951657
1,1994,2.607442
2,1995,2.80542
3,1996,2.931204
4,1997,2.33769


**Converting column data:**

Next, we must convert our column data types to be what we expect.
- Year --> int
- CPI_per_year --> float

In [13]:
cpi_per_year_us_df['Year'] = cpi_per_year_us_df['Year'].astype(int)
cpi_per_year_us_df['CPI_per_year'] = cpi_per_year_us_df[
    'CPI_per_year'].astype(float)

print('Columns names \'Year\' data type: ' + 
      str(cpi_per_year_us_df['Year'].dtype))
print('Columns names \'CPI_per_year\' data type: ' + 
      str(cpi_per_year_us_df['CPI_per_year'].dtype))

Columns names 'Year' data type: int64
Columns names 'CPI_per_year' data type: float64


**Currently, our columns *CPI_per_year* represents the annual inflation rate per year. We must take these values and convert it into the CPI rate from 1993 to 2024; this way, we can scale the costs to be in terms of nominal value in 2024 (what we consider our present day). We calculate this by doing the following:**

We set the base **Consumer Price Index (CPI) to be 2024 and thus CPI = 100**.
After setting the base as 2024 (CPI = 100), we can calculate the CPI for the years prior 2024 using the fomrula:

 ***CPI<sub>target_year</sub> = CPI<sub>prior_year</sub> * (1 + (inflation_rate<sub>current_year</sub> / 100))***

In [14]:

cpi_per_year_us_df.sort_values(by='Year', inplace=True)

multiply_by = 1 + (cpi_per_year_us_df['CPI_per_year'] / 100)

base_year = 1993
base_year_index = cpi_per_year_us_df[cpi_per_year_us_df[
    'Year'] == base_year]\
.index[0]

multiply_by.loc[base_year_index] = 1.0

cumulative_index = multiply_by.cumprod() * 100

cpi_per_year_us_df['CPI_per_year'] = cumulative_index

#Ensuring our changes worked
print("DataFrame after replacing rates with the index:")
print(cpi_per_year_us_df.head())


DataFrame after replacing rates with the index:
   Year  CPI_per_year
0  1993    100.000000
1  1994    102.607442
2  1995    105.486011
3  1996    108.578021
4  1997    111.116239


#### **Merging *flights_df* with *cpi_per_year_us_df* by *Year***
Here we are creating a **new** data frame to represent our *flights_df* with the addition of *cpi_per_year_us_df* per corresponding year. We Will do this through an **inner merge** on *flights_df* and *cpi_per_year_us_df* by year.

In [15]:
flights_and_inflation_df = pd.merge(flight_df, cpi_per_year_us_df,
                                     on='Year', how='inner')
flights_and_inflation_df.head()

,Year,quarter,nsmiles,passengers,fare,CPI_per_year
0,2021,3,970,180,81.43,187.576406
1,2021,3,970,19,208.93,187.576406
2,2021,3,580,204,184.56,187.576406
3,2021,3,580,264,182.64,187.576406
4,2021,3,328,398,177.11,187.576406


**Creating Inflation-Adjusted Fare Column:**

We now compute the fare adjusted to 2024 dollars using the CPI index we created earlier.

First, we identify the CPI value for 2024 (our base CPI = 100).

In [16]:
base_cpi = cpi_per_year_us_df.loc[cpi_per_year_us_df['Year'] == 2024, 'CPI_per_year'].iloc[0]

year_to_cpi = cpi_per_year_us_df.set_index('Year')['CPI_per_year']

original_cpi = flights_and_inflation_df['Year'].map(year_to_cpi)

flights_and_inflation_df['fare_with_inflation'] = np.round((flights_and_inflation_df['fare'] * (base_cpi / original_cpi)), 2)

print("Preview of inflation-adjusted fare:")
print(flights_and_inflation_df[['Year', 'fare', 'fare_with_inflation']].head())

Preview of inflation-adjusted fare:
   Year    fare  fare_with_inflation
0  2021   81.43                94.27
1  2021  208.93               241.87
2  2021  184.56               213.66
3  2021  182.64               211.43
4  2021  177.11               205.03


**Finding Missing Values:**\
For good practice, we will ensure there are no missing values in our dataset after this merge and column addition, to preserve our data usability.  

In [17]:
missing_values_after_merge = flights_and_inflation_df.isnull().sum()
print(missing_values_after_merge)

Year                   0
quarter                0
nsmiles                0
passengers             0
fare                   0
CPI_per_year           0
fare_with_inflation    0
dtype: int64


**Dropping the CPI Column:**

Since we have already calculated the inflation-adjusted fare (fare_with_inflation), the CPI_per_year column is no longer needed in the final dataset.

In [18]:
flights_and_inflation_df = flights_and_inflation_df.drop(columns=['CPI_per_year'])

print("Columns after dropping CPI_per_year:")
flights_and_inflation_df.head()

Columns after dropping CPI_per_year:


,Year,quarter,nsmiles,passengers,fare,fare_with_inflation
0,2021,3,970,180,81.43,94.27
1,2021,3,970,19,208.93,241.87
2,2021,3,580,204,184.56,213.66
3,2021,3,580,264,182.64,211.43
4,2021,3,328,398,177.11,205.03


**Converting Data to a new CSV:**\
After completing all necessary cleaning steps and confirming there are no remaining missing or invalid values, we export the final dataframe as a CSV file. This cleaned dataset will be used in our main analysis notebook.

In [19]:
flights_and_inflation_df.to_csv('flights.csv', index=False) 